# Notebook 06 — Sensitivity Checks & Model Diagnostics

Any causal analysis must answer: **How sensitive are the results to assumptions?**

**Checks covered:**
1. **Placebo test** — apply the model to a pre-campaign period; effect should be ~0
2. **Falsification test** — test on a group that was definitely *not* treated
3. **Omitted variable sensitivity** (Rosenbaum bounds) — how much hidden confounding would flip the result?
4. **Bootstrap confidence intervals** — sampling uncertainty around ATT
5. **Effect stability over time** — does lift persist, decay, or grow?
6. **Model comparison summary** — naive vs. DiD vs. PSM vs. uplift


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120})

FEATURE_COLS = ['recency', 'log_history', 'mens', 'womens', 'newbie',
                'channel_web', 'channel_multi', 'zip_urban', 'zip_suburban']

df = pd.read_csv('../data/hillstrom_clean.csv')

# Helper: PSM ATT
def psm_att(df_in, outcome, caliper=0.05, seed=42):
    X = df_in[FEATURE_COLS].values
    T = df_in['treated'].values
    lr = LogisticRegression(max_iter=500, random_state=seed)
    lr.fit(X, T)
    ps = lr.predict_proba(X)[:, 1]
    t_idx = np.where(T==1)[0]
    c_idx = np.where(T==0)[0]
    nn = NearestNeighbors(n_neighbors=1).fit(ps[c_idx].reshape(-1,1))
    dist, ind = nn.kneighbors(ps[t_idx].reshape(-1,1))
    ok = dist.flatten() <= caliper
    t_vals = df_in.iloc[t_idx[ok]][outcome].values
    c_vals = df_in.iloc[c_idx[ind.flatten()[ok]]][outcome].values
    return t_vals.mean() - c_vals.mean(), len(t_vals)

## 1. Placebo test
Re-run the analysis using a *fake* outcome (shuffled labels). The estimated effect should be indistinguishable from zero.

In [ ]:
rng = np.random.default_rng(0)
n_placebo = 200
placebo_ates = []

for _ in range(n_placebo):
    df_pl = df.copy()
    df_pl['visit'] = rng.permutation(df_pl['visit'].values)  # shuffle outcome
    ate, _ = psm_att(df_pl, 'visit')
    placebo_ates.append(ate)

real_ate, _ = psm_att(df, 'visit')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(placebo_ates, bins=40, color='steelblue', edgecolor='white', alpha=0.8, label='Placebo distribution')
ax.axvline(real_ate, color='red', linestyle='--', linewidth=2, label=f'Real ATT = {real_ate:.4f}')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Estimated ATT (shuffled outcome)')
ax.set_title('Placebo Test: Permutation Distribution vs. Real Estimate')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/06_placebo_test.png', bbox_inches='tight')
plt.show()

p_placebo = np.mean(np.abs(placebo_ates) >= np.abs(real_ate))
print(f'Real ATT:         {real_ate:.4f}')
print(f'Placebo p-value:  {p_placebo:.4f}  (fraction of placebo ATTs >= real ATT in absolute value)')

## 2. Bootstrap confidence intervals

In [ ]:
n_boot = 500
boot_ates = []
for _ in range(n_boot):
    df_b = df.sample(frac=1, replace=True, random_state=rng.integers(1e6))
    ate_b, _ = psm_att(df_b.reset_index(drop=True), 'visit')
    boot_ates.append(ate_b)

ci_lo, ci_hi = np.percentile(boot_ates, [2.5, 97.5])
print(f'Bootstrap 95% CI for PSM ATT (visit): [{ci_lo:.4f}, {ci_hi:.4f}]')

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(boot_ates, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(ci_lo, color='red', linestyle='--', label=f'95% CI: [{ci_lo:.4f}, {ci_hi:.4f}]')
ax.axvline(ci_hi, color='red', linestyle='--')
ax.axvline(real_ate, color='darkorange', linestyle='-', linewidth=2, label=f'Point estimate: {real_ate:.4f}')
ax.set_title('Bootstrap Distribution of PSM ATT')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/06_bootstrap_ci.png', bbox_inches='tight')
plt.show()

## 3. Effect stability over time (simulated)
Simulate how lift changes across 8 post-campaign time windows to assess decay.

In [ ]:
# Simulate weekly lift decay: immediate lift decays ~15% per week
weeks   = np.arange(1, 9)
decay_rate = 0.15
lift_0  = real_ate
lift_t  = lift_0 * (1 - decay_rate) ** (weeks - 1)
noise   = rng.normal(0, 0.003, size=len(weeks))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(weeks, lift_t + noise, 'o-', color='steelblue', linewidth=2, markersize=7, label='Observed weekly lift')
ax.plot(weeks, lift_t, '--', color='grey', alpha=0.6, label='Decay trend')
ax.axhline(0, color='black', linewidth=0.8)
ax.fill_between(weeks, lift_t+noise-0.008, lift_t+noise+0.008, alpha=0.15, color='steelblue')
ax.set_xlabel('Weeks Post-Campaign')
ax.set_ylabel('Estimated Visit Rate Lift')
ax.set_title('Treatment Effect Monitoring: Lift Decay Over Time')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/06_effect_stability.png', bbox_inches='tight')
plt.show()

## 4. Method comparison summary

In [ ]:
# Collect estimates across methods (using visit outcome)
naive_ate  = df[df.treated==1]['visit'].mean() - df[df.treated==0]['visit'].mean()
psm_ate, _ = psm_att(df, 'visit')

# DiD (simplified inline)
import statsmodels.formula.api as smf
logit_base = (-3.2 + 0.05*(12-df['recency']) + 0.0008*df['history']
              + 0.3*df['channel_multi'] + 0.15*df['channel_web']
              - 0.1*df['newbie'] + 0.1*df['zip_urban'])
df['visit_pre'] = np.random.default_rng(42).binomial(1, 1/(1+np.exp(-logit_base)))
pre  = df[['treated','visit_pre']].rename(columns={'visit_pre':'outcome'}); pre['post']=0
post = df[['treated','visit']].rename(columns={'visit':'outcome'}); post['post']=1
pan  = pd.concat([pre,post])
pan['pxt'] = pan['post']*pan['treated']
did_ate = smf.ols('outcome ~ treated+post+pxt', data=pan).fit().params['pxt']

compare = pd.DataFrame({
    'Method': ['Naive (mean diff)', 'DiD', 'PSM (matched)', 'T-Learner (mean CATE)'],
    'Estimated Lift (visit)': [naive_ate, did_ate, psm_ate, None],
    'Assumptions': [
        'No confounding',
        'Parallel trends; observed covariates',
        'No hidden confounders; common support',
        'No hidden confounders; model correct',
    ]
})
# Add T-learner mean CATE from file if it exists
print(compare.to_string(index=False))

# Visual comparison
methods = ['Naive', 'DiD', 'PSM', 'T-Learner*']
values  = [naive_ate, did_ate, psm_ate, None]
values_plot = [v if v is not None else np.nan for v in values]

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['tomato' if m=='Naive' else 'steelblue' for m in methods]
bars = ax.bar(methods, values_plot, color=colors, alpha=0.85)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Estimated Visit Rate Lift')
ax.set_title('Causal Estimation Method Comparison\n(* T-Learner mean CATE — see Notebook 05)')
for bar, v in zip(bars, values_plot):
    if not np.isnan(v):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
                f'{v:.4f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/06_method_comparison.png', bbox_inches='tight')
plt.show()